<a href="https://colab.research.google.com/github/zainkhan-dev/flyrank-ml-internship-week1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainkhan-dev/flyrank-ml-internship-week1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [10]:
import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [12]:
df_files = con.sql("""
SELECT *
FROM glob('hf://datasets/FlyRank/internship-warehouse/**')
""").df()

for f in df_files["file"]:
    print(f)

hf://datasets/FlyRank/internship-warehouse/.gitattributes
hf://datasets/FlyRank/internship-warehouse/README.md
hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
hf://datasets/FlyRank/internship-warehouse

In [13]:
con.sql("""
SELECT *
FROM glob('hf://datasets/FlyRank/internship-warehouse/*')
""").df()

,file
0,hf://datasets/FlyRank/internship-warehouse/.gitattributes
1,hf://datasets/FlyRank/internship-warehouse/README.md
2,hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
3,hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
4,hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet
5,hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet


In [14]:
# dim_content schema check
con.sql("""
DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [15]:
con.sql("""
DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [16]:
con.sql("""
SELECT content_hash_id, COUNT(*) AS c
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
GROUP BY content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,c


In [17]:
import pandas as pd

# --- Step 1: aggregate daily performance to one row per content item, March 2026 ---
agg = con.sql("""
SELECT
    content_hash_id,
    AVG(CASE WHEN gsc_data_available IS TRUE THEN gsc_avg_position END) AS avg_gsc_position,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS total_clicks,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) * 1.0
        / NULLIF(SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END), 0) AS ctr,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END), 0) AS engagement_rate,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_ai ELSE 0 END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END), 0) AS pct_ai_sessions,
    MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS has_gsc_data,
    MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_ga4_data
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
GROUP BY content_hash_id
""").df()

# --- Step 2: bring in content_type from dim_content (static, safe categorical) ---
dim_content = con.sql("""
SELECT content_hash_id, content_type
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""").df()

fv = agg.merge(dim_content, on="content_hash_id", how="left")

# --- Step 3: fill strategy (not drop) ---
# Numeric GSC/GA4 fields: fill with the column MEDIAN, not zero, so a
# missing-tracking item doesn't get treated as "zero position" or "zero engagement."
for col in ["avg_gsc_position", "ctr", "engagement_rate", "pct_ai_sessions"]:
    fv[col] = fv[col].fillna(fv[col].median())

# total_clicks: genuine zero is meaningful here (no clicks logged), so 0-fill is fine.
fv["total_clicks"] = fv["total_clicks"].fillna(0)

# has_gsc_data / has_ga4_data: already 0/1 flags from MAX(), no NaNs expected, but just in case:
fv["has_gsc_data"] = fv["has_gsc_data"].fillna(0)
fv["has_ga4_data"] = fv["has_ga4_data"].fillna(0)

# content_type: fill missing with an explicit "unknown" category rather than dropping.
fv["content_type"] = fv["content_type"].fillna("unknown")
fv = pd.get_dummies(fv, columns=["content_type"], prefix="type")

print(fv.shape)
fv.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 11)


,content_hash_id,avg_gsc_position,total_clicks,ctr,engagement_rate,pct_ai_sessions,has_gsc_data,has_ga4_data,type_comparison article,type_feedly article,type_keyword article
0,content_39d7361b4945d504,4.074107,0.0,0.000000,0.0,0.0,1,0,False,False,True
1,content_cec711b02f3bbde6,4.428747,4.0,0.006645,0.0,0.0,1,0,False,False,True
2,content_275b6f7f733016d4,4.866123,1.0,0.001235,0.0,0.0,1,0,False,False,True
3,content_ceaec531566ffcfc,8.978086,0.0,0.000000,0.0,0.0,1,0,False,False,True
4,content_755d951187fcd70a,1.854929,6.0,0.003229,0.0,0.0,1,0,False,False,True


In [18]:
fv[["engagement_rate", "pct_ai_sessions", "avg_gsc_position", "ctr"]].median()

,0
engagement_rate,0.000000
pct_ai_sessions,0.000000
avg_gsc_position,8.505296
ctr,0.000000


In [19]:
con.sql("""
SELECT
  is_published,
  is_deleted,
  COUNT(*) AS n,
  MIN(optimization_eligible_date) AS min_opt_elig,
  MAX(optimization_eligible_date) AS max_opt_elig
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
GROUP BY is_published, is_deleted
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,is_published,is_deleted,n,min_opt_elig,max_opt_elig
0,True,False,411540,2026-06-08,2026-08-20
1,False,False,6507,NaT,NaT
2,False,True,101559,NaT,NaT


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

**`avg_gsc_position`** — meaning: mean Search Console ranking position across March, computed only from rows where `gsc_data_available IS TRUE`. Missing: filled with the column median (8.51) for items with zero available GSC rows — a deliberate compromise that avoids the "zero-fill lies" trap (missing tracking ≠ position zero), but note this means "unknown" items get a plausible-but-fabricated typical value. Categorical: no. Available when: yes — it's a same-window historical average, nothing forward-looking.

**`total_clicks`** — meaning: sum of GSC clicks across March, `gsc_data_available`-gated, else 0. Missing: filled with 0 — a genuine zero (no clicks logged) is a meaningful, honest value here, unlike the rate columns. Categorical: no. Available when: yes — same-day logged counts, summed within the window.

**`ctr`** — meaning: `SUM(clicks) / SUM(impressions)`, GSC-gated. Missing: filled with the column median (0.0) — worth naming explicitly: because more than half of all items had zero clicks relative to impressions this month, the median-fill and a zero-fill turn out to be nearly identical here, which is a real fact about this data, not evidence the fill logic is doing nothing. Categorical: no. Available when: yes — same-window ratio of two same-window logged values.

**`engagement_rate`** — meaning: `SUM(engaged sessions) / SUM(sessions)`, GA4-gated. Missing: median-filled (0.0), same reasoning as `ctr` — most items show zero engaged sessions relative to total sessions this month. Categorical: no. Available when: yes when GA4 tracking is active; for the ~73% of items without GA4 data this month, the filled value is a stand-in, not an observation — flagged separately via `has_ga4_data`.

**`pct_ai_sessions`** — meaning: `SUM(AI-referral sessions) / SUM(sessions)`, GA4-gated. Missing: median-filled (0.0), same pattern as above. Categorical: no. Available when: same caveat as `engagement_rate` — genuinely available only when GA4 tracking exists; flagged via `has_ga4_data`.

**`has_gsc_data`** — meaning: binary flag, 1 if the item had at least one row with `gsc_data_available IS TRUE` in March, else 0. Missing: none — derived directly from `MAX()`, always populated. Categorical: effectively binary/boolean. Available when: yes — it's a same-window observability flag, not a performance metric itself. Purpose: lets a reviewer (or a future model) distinguish "real median" from "filled median" in the GSC columns above.

**`has_ga4_data`** — same as `has_gsc_data`, for GA4 availability. Given only ~4% of rows have GA4 data this month, this flag is doing real work — it's the difference between trusting `engagement_rate`/`pct_ai_sessions` and knowing they're filled.

**`content_type`** (one-hot encoded into `type_comparison article`, `type_feedly article`, `type_keyword article`, etc.) — meaning: static content classification from `dim_content`. Missing: filled with an explicit `"unknown"` category before encoding, rather than dropping rows or silently omitting a dummy column — keeps missing-type items visible instead of erasing them. Categorical: yes, one-hot encoded (no ordinal relationship between types, so one-hot is appropriate over integer-encoding). Available when: yes — content type is fixed metadata, not an outcome, and doesn't change based on performance.

**One honest limitation on the fill strategy overall:** median-filling turns "no data" into "typical data" for downstream math, which is safer than zero-filling but still means a clustering algorithm can't distinguish a genuinely median-performing item from an unknown one unless it's also given the `has_gsc_data`/`has_ga4_data` flags — and even then, nothing forces the algorithm to actually use those flags meaningfully. This is a real trade-off, not a solved problem.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [20]:
# --- Bring in the two candidate leakage columns ---
leak_source = con.sql("""
SELECT content_hash_id, is_deleted, optimization_eligible_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""").df()

fv_leak = fv.merge(leak_source, on="content_hash_id", how="left")

# Vector 1: label-derived column (reusing the ML-04 proxy trick)
fv_leak["proxy_high_performer"] = (
    fv_leak["avg_gsc_position"] < fv_leak["avg_gsc_position"].median()
).astype(int)

# Vector 2: future-window leak — days from March 31 to optimization eligibility
import pandas as pd
fv_leak["days_to_opt_eligible"] = (
    pd.to_datetime(fv_leak["optimization_eligible_date"]) - pd.Timestamp("2026-03-31")
).dt.days

# Vector 3: product-flag leak — is_deleted, encoded as int
fv_leak["is_deleted_flag"] = fv_leak["is_deleted"].astype(int)

fv_leak[["proxy_high_performer", "days_to_opt_eligible", "is_deleted_flag"]].describe()

,proxy_high_performer,days_to_opt_eligible,is_deleted_flag
count,331437.000000,42517.000000,331437.000000
mean,0.266624,114.938236,0.019581
std,0.442195,16.202675,0.138557
min,0.000000,69.000000,0.000000
25%,0.000000,101.000000,0.000000
50%,0.000000,117.000000,0.000000
75%,1.000000,128.000000,0.000000
max,1.000000,142.000000,1.000000


In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

honest_cols = ["avg_gsc_position", "total_clicks", "ctr", "engagement_rate", "pct_ai_sessions"]

def purity(crosstab):
    return crosstab.max(axis=1).sum() / crosstab.values.sum()

def run_cluster(df, feature_cols, label_col, label_name):
    sub = df.dropna(subset=feature_cols + [label_col]).copy()
    X = StandardScaler().fit_transform(sub[feature_cols])
    km = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X)
    ct = pd.crosstab(km.labels_, sub[label_col])
    p = purity(ct)
    print(f"--- {label_name} ---")
    print(f"n={len(sub)}  purity={p:.3%}")
    print(ct)
    print()
    return p

# ---- Vector 1: label-derived column ----
p_honest_1 = run_cluster(fv_leak, honest_cols, "proxy_high_performer", "HONEST (vector 1 baseline)")
p_leaky_1  = run_cluster(fv_leak, honest_cols + ["proxy_high_performer"], "proxy_high_performer", "LEAKY: label-derived column added")

# ---- Vector 2: future-window leak ----
fv_v2 = fv_leak.copy()
fv_v2["future_leak_bucket"] = (fv_v2["days_to_opt_eligible"] < fv_v2["days_to_opt_eligible"].median()).astype(int)
p_honest_2 = run_cluster(fv_v2, honest_cols, "future_leak_bucket", "HONEST (vector 2 baseline)")
p_leaky_2  = run_cluster(fv_v2, honest_cols + ["days_to_opt_eligible"], "future_leak_bucket", "LEAKY: future-window column added")

# ---- Vector 3: product-flag leak ----
p_honest_3 = run_cluster(fv_leak, honest_cols, "is_deleted_flag", "HONEST (vector 3 baseline)")
p_leaky_3  = run_cluster(fv_leak, honest_cols + ["is_deleted_flag"], "is_deleted_flag", "LEAKY: product flag added")

print("SUMMARY")
print(f"Vector 1 (label-derived):  honest={p_honest_1:.3%}  leaky={p_leaky_1:.3%}")
print(f"Vector 2 (future-window):  honest={p_honest_2:.3%}  leaky={p_leaky_2:.3%}")
print(f"Vector 3 (product-flag):   honest={p_honest_3:.3%}  leaky={p_leaky_3:.3%}")

--- HONEST (vector 1 baseline) ---
n=331437  purity=73.338%
proxy_high_performer       0      1
row_0                              
0                     241652  87152
1                       1416   1217

--- LEAKY: label-derived column added ---
n=331437  purity=99.958%
proxy_high_performer       0      1
row_0                              
0                     242929      0
1                        139  88369

--- HONEST (vector 2 baseline) ---
n=331437  purity=93.942%
future_leak_bucket       0      1
row_0                            
0                   309111  19693
1                     2246    387

--- LEAKY: future-window column added ---
n=42517  purity=94.421%
future_leak_bucket      0      1
row_0                           
0                      38  17746
1                   22399   2334

--- HONEST (vector 3 baseline) ---
n=331437  purity=98.042%
is_deleted_flag       0     1
row_0                        
0                322315  6489
1                  2632     1

--- LE

In [22]:
sub_v2 = fv_v2.dropna(subset=honest_cols + ["days_to_opt_eligible", "future_leak_bucket"]).copy()
p_honest_2_fair = run_cluster(sub_v2, honest_cols, "future_leak_bucket", "HONEST (vector 2, same n as leaky)")
p_leaky_2_fair  = run_cluster(sub_v2, honest_cols + ["days_to_opt_eligible"], "future_leak_bucket", "LEAKY (vector 2, same n)")

--- HONEST (vector 2, same n as leaky) ---
n=42517  purity=52.772%
future_leak_bucket      0      1
row_0                           
0                   17705  15971
1                    4732   4109

--- LEAKY (vector 2, same n) ---
n=42517  purity=94.421%
future_leak_bucket      0      1
row_0                           
0                      38  17746
1                   22399   2334



## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.